# Parse instruments
Version the reference data carried by checked FIX market messages.


In [ ]:
project_root = "."
# Checked FIX market messages, as `parse_fix` wrote them.
source = "fix.market"
start = None
end = None
fix_dictionary = "data/fix"
catalog = "rekep"
catalog_properties = {}
table_properties = {"history.expire.max-snapshot-age-ms": "604800000"}
branch = "root"
target = "market.instruments"
batch_row_size = 65_536
commit_row_size = 250_000
log_level = "INFO"


In [ ]:
from pyiceberg.expressions import And, GreaterThanOrEqual, In, LessThan

from rekep.fix.registry import FixRegistry
from rekep.iceberg import IcebergDataset
from rekep.logs import Stage, configure
from rekep.market import InstrumentUpdate
from rekep.text import FixMsg
from rekep.times import unix_of
from rekep.urls import Url

configure(log_level)


def _window(lower, upper, column="unix"):
    predicates = []
    if lower is not None:
        predicates.append(GreaterThanOrEqual(column, lower))
    if upper is not None:
        predicates.append(LessThan(column, upper))
    return None if not predicates else predicates[0] if len(predicates) == 1 else And(*predicates)


# The FIX stage resolved the transaction clock and wrote it as `unix`, which is
# what this table is partitioned from. The ordered read below asks explicitly
# for transaction time; physical files keep `hash` as their sole sort key.
lower, upper = unix_of(start), unix_of(end, upper=True)
if batch_row_size <= 0 or commit_row_size <= 0:
    raise ValueError("batch_row_size and commit_row_size must be positive")
registry = FixRegistry(
    cache_dir=Url.from_string(str(fix_dictionary)).resolve(project_root),
    announce=print,
)
field = FixMsg.into_field()
messages = IcebergDataset(
    field=field.with_name(source),
    catalog=catalog,
    properties=dict(catalog_properties),
    branch=branch,
)
instruments = IcebergDataset(
    field=InstrumentUpdate.into_field(target),
    catalog=catalog,
    properties=dict(catalog_properties),
    table_properties=dict(table_properties),
    branch=branch,
)
stage = Stage(
    "parse_instruments",
    sources={"market": source},
    targets={"instruments": target},
    window=(lower, upper),
)


In [ ]:
read = written = 0


def _observed():
    """Every update the window's messages describe, enriched per ticker."""
    if not messages.exists:
        return iter(())
    reader = messages.read_arrow_reader(
        field, row_filter=_window(lower, upper), order_by=("unix", "msgseqnum", "hash")
    )
    return InstrumentUpdate.from_fixmsgs(FixMsg.from_arrow_reader(reader), registry=registry)


def _stored(xhashes):
    """What the table already holds for the lifecycles in one batch."""
    if not instruments.exists or not xhashes:
        return {}
    reader = instruments.read_arrow_reader(
        InstrumentUpdate.into_field(), row_filter=In("xhash", xhashes)
    )
    return {row.xhash: row for row in InstrumentUpdate.from_arrow_reader(reader)}


def _versions():
    """Only what changes the table, one bounded lookup per batch."""
    global read, written
    for batch in InstrumentUpdate.into_arrow_reader(
        _observed(), batch_row_size=batch_row_size
    ):
        observed = list(InstrumentUpdate.from_arrow_reader(iter((batch,))))
        read += len(observed)
        stored = _stored(tuple(row.xhash for row in observed))
        changed = list(InstrumentUpdate.versioned(observed, stored))
        if changed:
            written += len(changed)
            yield InstrumentUpdate.into_arrow_batch(changed)


def _with_first(first, rest):
    yield first
    yield from rest


# `overwrite_arrow_reader` and not an append: one xhash holds one current row,
# and a version replaces it. Nothing is written when no batch changed, so a
# replay of an unchanged window commits no snapshot.
changes = iter(_versions())
first = next(changes, None)
if first is not None:
    instruments.overwrite_arrow_reader(
        _with_first(first, changes),
        InstrumentUpdate.into_field(),
        merge_by=True,
        commit_row_size=commit_row_size,
    )

stage.says("observed %d instruments, of which %d are new versions", read, written)
result = stage.finished(read=read, written=written)
result
